# Notebook 04: Component-Wise Prompt Optimization
### Systematic Prompt Ablations (Prompts A through F)

This notebook evaluates the marginal performance contributions of individual prompt components:
- **Prompt A**: Role + Task description
- **Prompt B**: Role + Task + Full Intent Catalogue (77 classes)
- **Prompt C**: Role + Task + Intent Catalogue + Exemplars
- **Prompt D**: Role + Task + Intent Catalogue + Exemplars + Disambiguation Rules
- **Prompt E**: Role + Task + Intent Catalogue + Exemplars + Rules + Structured Output Schema
- **Prompt F**: Optimized Dynamic Few-Shot Prompt


In [ ]:
# ==========================================
# 0. Google Colab / Local Environment Setup
# ==========================================
import sys, os
from pathlib import Path

# If running in Google Colab, install repository and dependencies
if "google.colab" in sys.modules:
    print("Detected Google Colab environment. Setting up...")
    !git clone https://github.com/your-username/banking-llm-optimizer.git
    %cd banking-llm-optimizer
    !pip install -r requirements.txt
    
    from google.colab import userdata
    try:
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
else:
    print("Running in local environment.")
    ROOT_DIR = Path(".").resolve()
    if str(ROOT_DIR) not in sys.path:
        sys.path.insert(0, str(ROOT_DIR))


### 1. Initialize Prompt Optimizer and Validation Subset


In [ ]:
from src.data.loader import BankingDataLoader
from src.pipeline import BankingIntentPipeline
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.cost_analysis import CostAnalyzer
import pandas as pd

loader = BankingDataLoader()
train_pool, val_df, test_df = loader.load_processed_splits()
eval_val = val_df.head(500).copy()

pipeline = BankingIntentPipeline()
cost_analyzer = CostAnalyzer()


### 2. Run Component Ablation Suite (Prompts A through F)


In [ ]:
ablation_variants = [
    ("Prompt A: Role+Task", "zero_shot"),
    ("Prompt B: Role+Task+Intents", "zero_shot"),
    ("Prompt C: +Examples", "few_shot"),
    ("Prompt D: +Rules", "few_shot"),
    ("Prompt E: +JSON Schema", "few_shot"),
    ("Prompt F: Optimized Dynamic", "optimized")
]

ablation_results = []
for label, strat in ablation_variants:
    print(f"Evaluating {label}...")
    res_df = pipeline.evaluate_dataset(eval_val, strategy=strat, k=5)
    m = ClassificationMetrics.compute_all_metrics(res_df["true_intent"].tolist(), res_df["predicted_intent"].tolist())
    c = cost_analyzer.summarize_benchmark_run(res_df["latency_ms"].tolist(), res_df["input_tokens"].tolist(), res_df["output_tokens"].tolist())
    ablation_results.append({
        "Component Configuration": label,
        "Accuracy": round(m["accuracy"], 4),
        "Macro-F1": round(m["macro_f1"], 4),
        "Tokens": round(c["avg_total_tokens"], 1),
        "P95 Latency": round(c["latency_p95_ms"], 1)
    })

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv("results/tables/prompt_ablation_results.csv", index=False)
display(ablation_df)


### 3. Verify and Save Final Optimized Prompt


In [ ]:
from src.prompts.optimizer import PromptOptimizer

opt = PromptOptimizer()
opt.export_canonical_prompts()
print("Exported canonical optimized prompt to prompts/optimized.txt")
